# RBI Digital Banking & Payment Data — Understanding and Validation

**Project:** SBI Financial Performance & Digital Banking Analytics  
**Stage:** Data Understanding & Preparation  
**Data Source:** Reserve Bank of India (RBI)  
**Reporting Period:** October 2021 – September 2026  
**Tools:** Python, Pandas, NumPy, Jupyter Notebook

## Project Overview

This notebook evaluates the structure, completeness, consistency, and reliability of consolidated RBI digital banking and payment datasets before downstream analysis.

The project focuses on four banking channels:

- National Electronic Funds Transfer (NEFT)
- Real Time Gross Settlement (RTGS)
- Mobile Banking
- Internet Banking

The broader analysis compares State Bank of India (SBI) with five peer banks: Axis Bank, Bank of Baroda, HDFC Bank, ICICI Bank, and Punjab National Bank.

The objective is to identify potential data-quality issues, assess their impact on analysis, and establish whether the datasets are suitable for financial and digital banking performance comparisons.

**Portfolio Disclosure:** This is an independent analytical portfolio project using publicly available RBI data and bank annual reports. It is not commissioned, affiliated with, or approved by State Bank of India.

## 1. Environment Setup and Data Loading

### Objective

Import the Python libraries required for data inspection and validation, and load the consolidated RBI workbook.

### Libraries Used

- **Pandas:** Data loading, manipulation, profiling, and validation.
- **NumPy:** Numerical operations and potential analytical calculations.
- **Matplotlib:** Data visualization capabilities for subsequent analysis.
- **Pathlib:** Platform-independent file path management.

### Data Loading Approach

The RBI workbook contains four worksheets, each representing a different banking channel.

The worksheets are loaded into separate Pandas DataFrames and organized into a dictionary to enable consistent validation across all four datasets.

Relative file paths are used to improve notebook portability.

**Expected Outcome:** All four datasets should load successfully without changing their original values.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [26]:
path = Path.cwd().parent / "02_Consolidated_Data" / "RBI_Digital_Banking_and_Payment_Data.xlsx"

rbi_data = pd.ExcelFile(path)
print(rbi_data.sheet_names)

['NEFT', 'RTGS', 'Mobile Banking', 'Internet Banking']


In [27]:
df_neft = pd.read_excel(path, sheet_name="NEFT")
df_rtgs = pd.read_excel(path, sheet_name="RTGS")
df_mb = pd.read_excel(path, sheet_name="Mobile Banking")
df_ib = pd.read_excel(path, sheet_name="Internet Banking")

## 2. Dataset Inventory and Structural Assessment

### Objective

Identify the number of records and columns in each dataset to establish the overall data structure and scope.

### Dataset Inventory

| Dataset | Records | Columns | Reporting Coverage |
|---|---:|---:|---|
| NEFT | 13,874 | 8 | 60 months |
| RTGS | 14,566 | 20 | 60 months |
| Mobile Banking | 32,111 | 7 | 60 months |
| Internet Banking | 6,103 | 7 | 55 months |
| **Total** | **66,654** | — | — |

### Findings

The consolidated RBI workbook contains 66,654 bank-month records distributed across four banking channels.

Mobile Banking accounts for approximately 48.2% of all records, followed by RTGS (21.9%), NEFT (20.8%), and Internet Banking (9.2%).

RTGS has the largest number of columns because it separately reports inward and outward transactions, interbank and customer components, and transaction-share measures.

### Business Interpretation

The four datasets differ in structure and coverage. Their row counts represent reporting observations, not transaction counts or market shares.

Consequently, a larger dataset does not necessarily indicate greater transaction activity.

Each channel must be analysed using its appropriate transaction volume, amount, customer, or share measures.

### Conclusion

The dataset structures are consistent with the recorded acquisition inventory and are suitable for further data-quality assessment.

In [61]:
datasets = {
    "NEFT": df_neft,
    "RTGS": df_rtgs,
    "Mobile Banking": df_mb,
    "Internet Banking": df_ib
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"Dataset: {name}")
    print('='*50)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])


Dataset: NEFT
Rows: 13874
Columns: 8

Dataset: RTGS
Rows: 14566
Columns: 20

Dataset: Mobile Banking
Rows: 32111
Columns: 7

Dataset: Internet Banking
Rows: 6103
Columns: 7


## 3. Data Type Inspection

### Objective

Examine the data types of all columns to determine whether identifiers, reporting periods, transaction volumes, monetary amounts, and percentage measures are stored in formats suitable for analysis.

### Findings

- Bank IDs, years, and months are stored as integers.
- Bank names are stored as text.
- Transaction amounts are stored as floating-point numbers.
- Several transaction volume and customer-count columns are stored as floating-point numbers.
- RTGS transaction-share measures are stored as floating-point numbers.

### Business Interpretation

Floating-point data types are appropriate for monetary amounts and percentage measures.

For count-labelled columns, a floating-point data type may result from missing values or fractional values present in the source data.

A floating-point data type alone does not establish that a column contains invalid records.

### Data Preparation Decision

No automatic data type conversion is performed during this stage.

Count-labelled fields containing fractional source values are preserved pending authoritative correction evidence.

This approach avoids introducing unsupported changes to the original data.

In [31]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print("="*50)

    print(df.dtypes)


DATASET: NEFT
ID                            int64
Bank Name                    object
Year                          int64
Month                         int64
Inward Volume                 int64
Inward Amount (₹ crore)     float64
Outward Volume                int64
Outward Amount (₹ crore)    float64
dtype: object

DATASET: RTGS
ID                                      int64
Bank Name                              object
Year                                    int64
Month                                   int64
Inward Interbank Volume               float64
Inward Customer Volume                float64
Inward Total Volume                   float64
Inward Volume Share (%)               float64
Inward Interbank Amount (₹ crore)     float64
Inward Customer Amount (₹ crore)      float64
Inward Total Amount (₹ crore)         float64
Inward Amount Share (%)               float64
Outward Interbank Volume              float64
Outward Customer Volume               float64
Outward Total Volume    

## 4. Missing Value Assessment

### Objective

Identify missing observations, measure their frequency, and assess potential implications for banking performance analysis.

### Findings

**NEFT**

No missing values were detected.

**RTGS**

Missing values were identified across 16 numeric fields.

Notable examples include:

| Field | Missing Records | Missing Percentage |
|---|---:|---:|
| Inward Interbank Volume | 2,647 | 18.17% |
| Outward Interbank Volume | 6,364 | 43.69% |
| Inward Customer Volume | 89 | 0.61% |
| Outward Customer Volume | 731 | 5.02% |
| Inward Total Volume | 77 | 0.53% |
| Outward Total Volume | 250 | 1.72% |

Corresponding amount fields show the same missing-record counts as their associated volume fields.

**Mobile Banking**

The Active Customers column contains 2,246 missing records, representing 6.99% of Mobile Banking observations.

**Internet Banking**

No missing values were detected in the available consolidated records.

### Business Insights

1. NEFT has the strongest observed completeness among the four datasets.

2. RTGS interbank transaction analysis faces substantial coverage limitations, particularly for outward interbank transactions.

3. RTGS total transaction measures have considerably lower missingness than certain segment-level measures. This makes total-level analysis potentially more suitable than detailed interbank comparisons, subject to reporting coverage.

4. Missing Mobile Banking active-customer observations may affect customer adoption and transactions-per-customer analysis.

5. Internet Banking has no missing cells within its available records, but its reporting period is shorter than that of the other three channels.

### Data Handling Decision

Missing values will remain unchanged.

No missing transaction volume, amount, or customer-count value will be replaced with zero without evidence that zero represents the actual reported observation.

Analytical outputs using incomplete fields must disclose the relevant coverage limitations.

### Conclusion

The datasets remain usable for analysis, but completeness differs by channel and metric. These differences must be considered when interpreting comparative results.

In [33]:
for name, df in datasets.items():

    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print("="*50)

    missing = df.isnull().sum()

    if missing.sum() == 0:
        print("No missing values found!")

    else:
        result = pd.DataFrame({
            "Missing Values": missing,
            "Missing Percentage (%)": (
                df.isnull().mean() * 100
            ).round(2)
        })

        print(result[result["Missing Values"] > 0])


DATASET: NEFT
No missing values found!

DATASET: RTGS
                                    Missing Values  Missing Percentage (%)
Inward Interbank Volume                       2647                   18.17
Inward Customer Volume                          89                    0.61
Inward Total Volume                             77                    0.53
Inward Volume Share (%)                         77                    0.53
Inward Interbank Amount (₹ crore)             2647                   18.17
Inward Customer Amount (₹ crore)                89                    0.61
Inward Total Amount (₹ crore)                   77                    0.53
Inward Amount Share (%)                         77                    0.53
Outward Interbank Volume                      6364                   43.69
Outward Customer Volume                        731                    5.02
Outward Total Volume                           250                    1.72
Outward Volume Share (%)                     

## 5. Duplicate Record Validation

### Objective

Identify exact duplicate rows and duplicate bank-month records that could distort transaction totals or bank-level comparisons.

### Validation Rules

Two checks are performed:

1. **Exact duplicate check:** Identifies completely identical records.
2. **Bank-month key check:** Identifies repeated combinations of Bank ID, Year, and Month.

### Results

| Dataset | Exact Duplicates | Duplicate Bank-Month Keys |
|---|---:|---:|
| NEFT | 0 | 0 |
| RTGS | 0 | 0 |
| Mobile Banking | 0 | 0 |
| Internet Banking | 0 | 0 |

### Business Interpretation

No duplicate records were detected under the defined validation rules.

This reduces the risk of double-counting observations when aggregating transaction measures by bank or reporting period.

### Data Preparation Decision

No deduplication is required.

### Conclusion

All four datasets passed the duplicate-record validation checks.

In [36]:
for name, df in datasets.items():

    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print("="*50)

    # Check exact duplicate rows
    exact_duplicates = df.duplicated().sum()

    # Check duplicate bank-month records
    key_duplicates = df.duplicated(
        subset=["ID", "Year", "Month"]
    ).sum()

    print("Exact Duplicate Rows:", exact_duplicates)
    print("Duplicate Bank-Month Records:", key_duplicates)

    if exact_duplicates == 0 and key_duplicates == 0:
        print("No duplicate records found.")
    else:
        print("Duplicate records require investigation.")


DATASET: NEFT
Exact Duplicate Rows: 0
Duplicate Bank-Month Records: 0
No duplicate records found.

DATASET: RTGS
Exact Duplicate Rows: 0
Duplicate Bank-Month Records: 0
No duplicate records found.

DATASET: Mobile Banking
Exact Duplicate Rows: 0
Duplicate Bank-Month Records: 0
No duplicate records found.

DATASET: Internet Banking
Exact Duplicate Rows: 0
Duplicate Bank-Month Records: 0
No duplicate records found.


## 6. Reporting Period Validation

### Objective

Check the range of reporting years and verify that month values fall within the valid calendar range of 1 to 12.

### Results

| Dataset | Year Range | Month Range | Invalid Month Records |
|---|---|---|---:|
| NEFT | 2021–2026 | 1–12 | 0 |
| RTGS | 2021–2026 | 1–12 | 0 |
| Mobile Banking | 2021–2026 | 1–12 | 0 |
| Internet Banking | 2022–2026 | 1–12 | 0 |

### Findings

No invalid month numbers were detected in any dataset.

Internet Banking begins in 2022, consistent with the documented availability of RBI Internet Banking source observations from March 2022.

### Business Interpretation

Valid month numbers are necessary for reliable time-based grouping and trend analysis.

The documented reporting periods remain:

- NEFT, RTGS, and Mobile Banking: October 2021 to September 2026.
- Internet Banking: March 2022 to September 2026.

### Conclusion

The datasets passed the calendar-month validity check.

In [38]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print("="*50)
    print("Year Range:", df["Year"].min(), "-", df["Year"].max())
    print("Month Range:", df["Month"].min(), "-", df["Month"].max())

    invalid_months = df[~df["Month"].between(1, 12)]

    print("Invalid Month Records:", len(invalid_months))


DATASET: NEFT
Year Range: 2021 - 2026
Month Range: 1 - 12
Invalid Month Records: 0

DATASET: RTGS
Year Range: 2021 - 2026
Month Range: 1 - 12
Invalid Month Records: 0

DATASET: Mobile Banking
Year Range: 2021 - 2026
Month Range: 1 - 12
Invalid Month Records: 0

DATASET: Internet Banking
Year Range: 2022 - 2026
Month Range: 1 - 12
Invalid Month Records: 0


## 7. Bank Identifier Consistency

### Objective

Determine whether bank IDs and bank names are consistently associated within each RBI dataset.

### Validation Rules

- One bank ID should not identify multiple bank names.
- One bank name should not be associated with multiple bank IDs.

### Results

| Dataset | ID Conflicts | Bank Name Conflicts |
|---|---:|---:|
| NEFT | 0 | 0 |
| RTGS | 0 | 0 |
| Mobile Banking | 0 | 0 |
| Internet Banking | 0 | 0 |

### Business Interpretation

No conflicting ID-to-name or name-to-ID mappings were identified within the individual consolidated RBI datasets.

Consistent identifiers reduce the risk of attributing transaction activity to the wrong institution.

### Conclusion

All four RBI datasets passed the internal bank identifier consistency assessment.

In [40]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print('='*50)
    # Check whether one ID belongs to multiple bank names
    id_check = df.groupby("ID")["Bank Name"].nunique()

    # Check whether one bank name has multiple IDs
    bank_check = df.groupby("Bank Name")["ID"].nunique()
    
    print("ID conflicts:", (id_check > 1).sum())

    print("Bank Name conflicts:", (bank_check > 1).sum())


DATASET: NEFT
ID conflicts: 0
Bank Name conflicts: 0

DATASET: RTGS
ID conflicts: 0
Bank Name conflicts: 0

DATASET: Mobile Banking
ID conflicts: 0
Bank Name conflicts: 0

DATASET: Internet Banking
ID conflicts: 0
Bank Name conflicts: 0


## 8. Numerical Value and Percentage Validation

### Objective

Identify negative transaction metrics and percentage-share values outside their expected numerical ranges.

### Validation Rules

- Transaction volumes and transaction amounts are expected to be non-negative.
- RTGS percentage-share values are expected to fall between 0 and 100.

### Results

| Dataset | Negative Metric Values |
|---|---:|
| NEFT | 0 |
| RTGS | 0 |
| Mobile Banking | 0 |
| Internet Banking | 0 |

All four RTGS share fields also returned zero out-of-range observations.

### Business Interpretation

No negative transaction measures or out-of-range RTGS shares were identified.

These results support the numerical plausibility of the reported metrics.

However, passing range checks does not independently establish that all reported values are accurate or correctly sourced.

### Conclusion

All four datasets passed the specified numerical range validations. No corrections were required.

In [42]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print('='*50)
    #  # Check negative values in the dataset
    metric_cols = [
        col for col in df.columns
        if col not in ["ID", "Bank Name", "Year", "Month"]
    ]
    negative_values = (df[metric_cols] < 0).sum().sum()

    print("Negative Values:", negative_values)

    # Check percentage values
    percentage_cols = [
        col for col in df.columns
        if "Share (%)" in col
    ]

    for col in percentage_cols:

        invalid_percentages = (
            (df[col] < 0) | (df[col] > 100)
        ).sum()

        print(f"{col} - Invalid Values:", invalid_percentages)


DATASET: NEFT
Negative Values: 0

DATASET: RTGS
Negative Values: 0
Inward Volume Share (%) - Invalid Values: 0
Inward Amount Share (%) - Invalid Values: 0
Outward Volume Share (%) - Invalid Values: 0
Outward Amount Share (%) - Invalid Values: 0

DATASET: Mobile Banking
Negative Values: 0

DATASET: Internet Banking
Negative Values: 0


## 9. RTGS Component Reconciliation

### Objective

Assess the internal consistency of RTGS transaction totals by comparing reported totals with their corresponding interbank and customer components.

### Reconciliation Rules

**Inward Transaction Volume**

Inward Interbank Volume + Inward Customer Volume = Inward Total Volume

**Outward Transaction Volume**

Outward Interbank Volume + Outward Customer Volume = Outward Total Volume

**Inward Transaction Amount**

Inward Interbank Amount + Inward Customer Amount = Inward Total Amount

**Outward Transaction Amount**

Outward Interbank Amount + Outward Customer Amount = Outward Total Amount

A numerical tolerance of 0.000001 is applied when evaluating differences.

### Results

| Reconciliation | Detected Mismatches |
|---|---:|
| Inward Volume | 0 |
| Outward Volume | 0 |
| Inward Amount | 0 |
| Outward Amount | 0 |

### Business Interpretation

No reconciliation mismatches were detected among records with available component and total values, using the selected numerical tolerance.

This provides evidence of internal consistency in the evaluated RTGS records.

### Validation Limitation

Records containing missing reconciliation components are not evaluated by the current comparison because missing values propagate through arithmetic operations.

The notebook therefore does not establish complete reconciliation of every RTGS record.

### Conclusion

The evaluated RTGS records passed the four component reconciliation checks. Missing-data coverage remains a documented limitation.

In [44]:
df = df_rtgs

# Inward volume reconciliation
inward_volume_diff = (
    df["Inward Interbank Volume"]
    + df["Inward Customer Volume"]
    - df["Inward Total Volume"]
)

# Outward volume reconciliation
outward_volume_diff = (
    df["Outward Interbank Volume"]
    + df["Outward Customer Volume"]
    - df["Outward Total Volume"]
)

# Inward amount reconciliation
inward_amount_diff = (
    df["Inward Interbank Amount (₹ crore)"]
    + df["Inward Customer Amount (₹ crore)"]
    - df["Inward Total Amount (₹ crore)"]
)

# Outward amount reconciliation
outward_amount_diff = (
    df["Outward Interbank Amount (₹ crore)"]
    + df["Outward Customer Amount (₹ crore)"]
    - df["Outward Total Amount (₹ crore)"]
)

# Count mismatches
print("Inward Volume Mismatches:",
      (inward_volume_diff.abs() > 0.000001).sum())

print("Outward Volume Mismatches:",
      (outward_volume_diff.abs() > 0.000001).sum())

print("Inward Amount Mismatches:",
      (inward_amount_diff.abs() > 0.000001).sum())

print("Outward Amount Mismatches:",
      (outward_amount_diff.abs() > 0.000001).sum())

Inward Volume Mismatches: 0
Outward Volume Mismatches: 0
Inward Amount Mismatches: 0
Outward Amount Mismatches: 0


## 10. Six-Bank Cross-Dataset Consistency

### Objective

Verify that the six banks included in the financial performance comparison use consistent identifiers across the four RBI datasets and the annual financial KPI master.

### Banks Included

| Bank | Consistent Bank ID |
|---|---:|
| State Bank of India | 894 |
| Axis Bank | 73 |
| Bank of Baroda | 96 |
| HDFC Bank | 295 |
| ICICI Bank | 310 |
| Punjab National Bank | 676 |

### Findings

All six comparison banks have matching identifiers across:

- NEFT
- RTGS
- Mobile Banking
- Internet Banking
- Annual Financial KPI Master

No cross-dataset identifier inconsistencies were detected for the selected six banks.

### Business Interpretation

Consistent bank identifiers support reliable institution-level filtering, comparison, and integration across financial and digital banking datasets.

This is particularly important because the RBI datasets contain monthly observations while the financial KPI master contains annual observations.

The different reporting frequencies must still be handled appropriately during subsequent analysis.

### Conclusion

The six-bank cross-dataset consistency validation passed.

In [46]:

KPI_data = (
    Path.cwd().parent / "02_Consolidated_Data"  / "Bank_Financial_Performance_KPI_Data.xlsx"
)
df_KPI = pd.read_excel(KPI_data)

print("Shape:", df_KPI.shape)
print("\nColumn Names:", df_KPI.columns.tolist())
print("\nFirst 5 Records:")
df_KPI.head()

Shape: (30, 26)

Column Names: ['ID', 'Bank', 'Financial Year', 'Total Assets (₹ crore)', 'Total Deposits (₹ crore)', 'Gross Advances (₹ crore)', 'Net Interest Income (NII) (₹ crore)', 'Operating Profit (₹ crore)', 'Net Profit (PAT) (₹ crore)', 'Net Interest Margin (NIM) (%)', 'Return on Assets (ROA) (%)', 'Gross NPA (GNPA) Ratio (%)', 'Net NPA (NNPA) Ratio (%)', 'Total Assets Growth (%)', 'Deposit Growth (%)', 'Gross Advances Growth (%)', 'NII Growth (%)', 'Operating Profit Growth (%)', 'Net Profit Growth (%)', 'CASA Ratio (%)', 'Capital Adequacy Ratio (CRAR) (%)', 'Common Equity Tier 1 (CET1) Ratio (%)', 'Provision Coverage Ratio (PCR) (%)', 'Cost-to-Income Ratio (%)', 'Earnings Per Share (EPS) (₹/share)', 'Book Value Per Share (₹/share)']

First 5 Records:


,ID,Bank,Financial Year,Total Assets (₹ crore),Total Deposits (₹ crore),Gross Advances (₹ crore),Net Interest Income (NII) (₹ crore),Operating Profit (₹ crore),Net Profit (PAT) (₹ crore),Net Interest Margin (NIM) (%),...,NII Growth (%),Operating Profit Growth (%),Net Profit Growth (%),CASA Ratio (%),Capital Adequacy Ratio (CRAR) (%),Common Equity Tier 1 (CET1) Ratio (%),Provision Coverage Ratio (PCR) (%),Cost-to-Income Ratio (%),Earnings Per Share (EPS) (₹/share),Book Value Per Share (₹/share)
0,894,SBI,FY2021-22,4987597.0,4051534.0,2818000.0,120708.0,75292.0,31676.0,3.12,...,0.090308,0.052240,0.551984,45.28,13.83,9.94,90.20,53.31,35.49,269.48
1,894,SBI,FY2022-23,5516979.0,4423778.0,3269000.0,144841.0,83713.0,50232.0,3.37,...,0.199929,0.111845,0.585806,43.80,14.68,10.27,91.91,53.87,56.29,309.89
2,894,SBI,FY2023-24,6179694.0,4916077.0,3768000.0,159876.0,93797.0,61077.0,3.28,...,0.103803,0.120459,0.215898,41.11,14.28,10.36,91.89,55.66,68.44,363.84
3,894,SBI,FY2024-25,6676053.0,5382190.0,4221000.0,166340.0,110579.0,70901.0,3.09,...,0.040431,0.178918,0.160846,39.97,14.25,10.81,92.08,51.64,79.44,435.95
4,894,SBI,FY2025-26,7623012.0,5975642.0,4933000.0,173120.0,123015.0,80032.0,2.91,...,0.040760,0.112463,0.128785,39.46,15.40,12.29,91.97,50.11,87.59,512.31


In [47]:
comparison_banks = [
    "SBI", "Axis Bank", "Bank of Baroda",
    "HDFC Bank", "ICICI Bank", "Punjab National Bank"
]

all_datasets = {
    "NEFT": df_neft,
    "RTGS": df_rtgs,
    "Mobile Banking": df_mb,
    "Internet Banking": df_ib,
    "Financial KPI": df_KPI
}

for name, df in all_datasets.items():
    print(f"\n{'='*50}")
    print("DATASET:", name)
    print('='*50)

    bank_col = "Bank" if name == "Financial KPI" else "Bank Name"

    for bank in comparison_banks:

        ids = df[df[bank_col] == bank]["ID"].unique()

        print(bank, ":", ids)


DATASET: NEFT
SBI : [894]
Axis Bank : [73]
Bank of Baroda : [96]
HDFC Bank : [295]
ICICI Bank : [310]
Punjab National Bank : [676]

DATASET: RTGS
SBI : [894]
Axis Bank : [73]
Bank of Baroda : [96]
HDFC Bank : [295]
ICICI Bank : [310]
Punjab National Bank : [676]

DATASET: Mobile Banking
SBI : [894]
Axis Bank : [73]
Bank of Baroda : [96]
HDFC Bank : [295]
ICICI Bank : [310]
Punjab National Bank : [676]

DATASET: Internet Banking
SBI : [894]
Axis Bank : [73]
Bank of Baroda : [96]
HDFC Bank : [295]
ICICI Bank : [310]
Punjab National Bank : [676]

DATASET: Financial KPI
SBI : [894]
Axis Bank : [73]
Bank of Baroda : [96]
HDFC Bank : [295]
ICICI Bank : [310]
Punjab National Bank : [676]


## 11. Data Quality Assessment Summary

The data-understanding and validation process examined four RBI banking datasets containing 66,654 records.

### Key Findings

| Quality Dimension | Assessment |
|---|---|
| Structural integrity | Expected row and column counts confirmed |
| Data types | Inspected and documented |
| Completeness | Source-related gaps identified in RTGS and Mobile Banking |
| Duplicate records | No duplicates detected |
| Temporal validity | No invalid month numbers detected |
| Identifier consistency | No internal bank ID/name conflicts detected |
| Numerical validity | No negative transaction metrics or out-of-range RTGS shares detected |
| RTGS reconciliation | No mismatches among evaluable records |
| Cross-dataset bank identification | Six comparison-bank IDs aligned across five datasets |

### Analytical Implications

**NEFT:** Suitable for transaction volume and value analysis, with no missing cells or duplicate records detected.

**RTGS:** Suitable for total transaction analysis with documented missingness. Segment-level analysis requires additional care because some interbank and customer fields have substantial missing-data rates.

**Mobile Banking:** Suitable for transaction activity analysis. Active-customer comparisons must account for missing observations.

**Internet Banking:** Suitable for analysis of the available reporting period, beginning March 2022. Earlier observations must not be treated as zero.

### Overall Assessment

The datasets demonstrate satisfactory structural consistency for the planned analytical work, subject to documented source and coverage limitations.

No automatic imputation, rounding, deduplication, or unsupported value correction was performed.

## 12. Data Limitations and Treatment Decisions

### 12.1 Missing Source Observations

Missing RTGS component values and Mobile Banking active-customer observations are retained as missing rather than replaced with assumed values.

### 12.2 Internet Banking Reporting Coverage

Internet Banking data begins in March 2022. October 2021 through February 2022 are outside the available consolidated source coverage.

### 12.3 Fractional Count-Labelled Values

Earlier source review identified three fractional values in fields labelled as counts:

- Assam Gramin Vikash Bank, June 2022 — Mobile Banking Transaction Volume: 793,756.5
- Axis Bank, April 2023 — Mobile Banking Transaction Volume: 444,705,902.4296
- Baroda Gujarat Gramin Bank, September 2024 — Internet Banking Active Customers: 2,861.235

These values matched the corresponding source cells during the earlier review and were retained without rounding or correction.

The current notebook does not independently reproduce those source comparisons.

### 12.4 Reconciliation Coverage

RTGS component reconciliation excludes records where required component values are unavailable.

Therefore, zero detected mismatches should not be interpreted as complete validation of every RTGS observation.

### 12.5 Reporting Coverage and Bank Participation

Bank participation varies across payment channels and reporting months.

The presence of a bank in a dataset does not imply that it reported activity in every month.

### 12.6 Source Accuracy

The notebook performs structural and internal-consistency checks.

It does not establish a complete record-by-record reconciliation of the consolidated RBI datasets against all original monthly source workbooks.

### 12.7 Data Preservation

No source values were modified during the validation exercise.

The original numerical representations and missing values were preserved to maintain transparency and traceability.

## 13. Conclusion and Data Readiness

The RBI data-understanding exercise established the structure, completeness, identifier consistency, temporal validity, numerical plausibility, and selected internal reconciliation properties of the four consolidated banking datasets.

The analysis identified no duplicate bank-month records, internal bank identifier conflicts, negative transaction metrics, or out-of-range RTGS percentage shares.

The six selected comparison banks also demonstrated consistent identifiers across the RBI datasets and the financial KPI master.

The principal limitations concern missing RTGS segment observations, missing Mobile Banking active-customer values, shorter Internet Banking reporting coverage, and source-reported fractional values in certain count-labelled fields.

These limitations do not automatically invalidate the datasets, but they must be considered when selecting measures and interpreting results.

**Final Readiness Assessment:** The RBI datasets are suitable for proceeding to the planned analytical stage with documented limitations and metric-specific coverage considerations.

The next activity is to validate the annual financial KPI master before integrating financial and digital banking performance insights.